# Session validity check
Pour chaque session : aires disponibles + n neurones par aire, structure des stimuli, lick times.

In [28]:
import glob, os

# ── Scan automatique de data/brut/ ─────────────────────────────────────────
BRUT_DIR = 'data/brut'
SESSION_PATHS = sorted(glob.glob(os.path.join(BRUT_DIR, '**', '*.nwb'), recursive=True))

print(f'{len(SESSION_PATHS)} fichier(s) .nwb trouvé(s) dans {BRUT_DIR}/ :')
for p in SESSION_PATHS:
    print(f'  {p}')

25 fichier(s) .nwb trouvé(s) dans data/brut/ :
  data/brut/AO027_20181101_134512.nwb
  data/brut/AO028_20181102_110339.nwb
  data/brut/AO028_20181103_83222.nwb
  data/brut/AO035_20190507_225532.nwb
  data/brut/AO035_20190508_182649.nwb
  data/brut/AO036_20190507_183647.nwb
  data/brut/AO036_20190508_150856.nwb
  data/brut/AO039_20190627_151857.nwb
  data/brut/AO040_20190709_102521.nwb
  data/brut/AO049_20191009_160435.nwb
  data/brut/AO049_20191010_182347.nwb
  data/brut/AO050_20191007_195438.nwb
  data/brut/AO050_20191008_124336.nwb
  data/brut/AO051_20191004_182709.nwb
  data/brut/AO065_20201023_100155.nwb
  data/brut/AO066_20201023_180037.nwb
  data/brut/AO067_20201022_190740.nwb
  data/brut/AO074_20201222_164350.nwb
  data/brut/AO075_20201223_134131.nwb
  data/brut/AO076_20201222_212759.nwb
  data/brut/AO077_20201223_91923.nwb
  data/brut/AO078_20201216_230018.nwb
  data/brut/AO078_20201217_213650.nwb
  data/brut/AO079_20201212_163030.nwb
  data/brut/AO079_20201214_203254.nwb


In [29]:
import numpy as np
import pandas as pd
from pynwb import NWBHDF5IO
from collections import Counter

# ── Helpers d'affichage ─────────────────────────────────────────────────────
GRN = '\033[92m'; YLW = '\033[93m'; RED = '\033[91m'; BLD = '\033[1m'; RST = '\033[0m'
HDR = lambda s: print(f'\n{BLD}{s}{RST}')
OK  = lambda s: print(f'    {GRN}{s}{RST}')
WRN = lambda s: print(f'    {YLW}{s}{RST}')
ERR = lambda s: print(f'    {RED}{s}{RST}')

# ────────────────────────────────────────────────────────────────────────────
for path in SESSION_PATHS:
    print(f'\n{BLD}{"="*72}{RST}')
    print(f'{BLD}  {path}{RST}')
    print(f'{BLD}{"="*72}{RST}')

    try:
        with NWBHDF5IO(path, 'r') as io:
            nwb = io.read()

            # ── 1. AIRES & NEURONES ──────────────────────────────────────────
            HDR('[1] Aires disponibles & nombre de neurones')
            units_df = nwb.units.to_dataframe()
            print(f'    Unités totales : {len(units_df)}')

            # Détecte la colonne d'aire selon le format
            for col_candidate in ('Target_area', 'ccf_parent_acronym', 'ccf_acronym',
                                  'brain_area', 'location'):
                if col_candidate in units_df.columns:
                    area_col = col_candidate
                    break
            else:
                area_col = None

            if area_col:
                print(f'    Colonne aire   : {area_col}')
                raw = units_df[area_col].values
                areas = np.array([a.decode() if isinstance(a, (bytes, bytearray)) else str(a)
                                   for a in raw])
                counts = Counter(areas)
                wS1_synonyms = {'SSp-bfd', 'wS1', 'BC', 'S1BF'}
                print(f'    {len(counts)} aire(s) :')
                for aire, n in sorted(counts.items(), key=lambda x: -x[1]):
                    marker = f'{GRN}★{RST}' if aire in wS1_synonyms or 'SSp' in aire else ' '
                    print(f'      {marker} {aire:35s} {n:3d} neurones')
                print(f'    (★ = probable wS1 / SSp-bfd)')
            else:
                ERR(f'Aucune colonne d\'aire trouvée.')
                print(f'    Colonnes dispo : {list(units_df.columns)}')

            # ── 2. STRUCTURE DES STIMULI ─────────────────────────────────────
            HDR('[2] Structure des stimuli')
            tr = nwb.trials.to_dataframe()
            print(f'    Trials totaux : {len(tr)}')

            # — Whisker stim —
            if 'whisker_stim_amplitude' in tr.columns:
                vals   = tr['whisker_stim_amplitude'].dropna()
                amps   = np.sort(vals[vals > 0].unique())
                n_whisk = len(vals[vals > 0])
                if len(amps) > 1:
                    print(f'\n    Whisker stim  : {GRN}{len(amps)} amplitudes{RST} → {amps}')
                else:
                    print(f'\n    Whisker stim  : {YLW}amplitude unique{RST} → {amps}')
                print(f'    Trials whisker : {n_whisk}')
                for a in amps:
                    n = int((tr['whisker_stim_amplitude'] == a).sum())
                    print(f'      amp {a:.0f} : {n:5d} trials')

            elif 'whisker_stim' in tr.columns:
                col = tr['whisker_stim']
                try:
                    n_w = int(col.astype(float).sum())
                except Exception:
                    n_w = col.notna().sum()
                print(f'\n    Whisker stim  : {YLW}binaire (0/1) — stimulus unique, pas d\'amplitude{RST}')
                print(f'    Trials whisker : {n_w}')

            elif 'whisker_stim_time' in tr.columns:
                n_w = tr['whisker_stim_time'].notna().sum()
                print(f'\n    Whisker stim  : colonne whisker_stim_time ({n_w} trials)')

            else:
                WRN('Aucune colonne whisker_stim trouvée.')

            # — Auditory stim —
            for aud_col in ('auditory_stim', 'auditory_stim_amplitude', 'auditory_stim_time'):
                if aud_col in tr.columns:
                    try:
                        n_a = int(tr[aud_col].astype(float).sum())
                    except Exception:
                        n_a = tr[aud_col].notna().sum()
                    print(f'\n    Auditory stim : {n_a} trials  (colonne \'{aud_col}\')')
                    break

            # — Catch / no-stim —
            for catch_col in ('no_stim', 'catch', 'no_stim_time'):
                if catch_col in tr.columns:
                    try:
                        n_c = int(tr[catch_col].astype(float).sum())
                    except Exception:
                        n_c = tr[catch_col].notna().sum()
                    print(f'    Catch         : {n_c} trials  (colonne \'{catch_col}\')')
                    break

            # — Résumé rapide des colonnes —
            print(f'\n    Toutes les colonnes trials : {list(tr.columns)}')

            # ── 3. LICK TIMES ────────────────────────────────────────────────
            HDR('[3] Lick times disponibles')
            try:
                beh = nwb.processing['behavior']
                found_any = False

                # PiezoLickSignal (signal continu)
                try:
                    ps = beh['BehavioralTimeSeries']['PiezoLickSignal']
                    sr = round(1.0 / float(np.median(np.diff(np.array(ps.timestamps[:500])))))
                    OK(f'PiezoLickSignal (continu) : {len(ps.data)} pts @ ~{sr} Hz')
                    found_any = True
                except Exception:
                    pass

                # BehavioralEvents
                try:
                    be = beh['BehavioralEvents']
                    for key in be.time_series:
                        obj = be[key]
                        ts  = np.array(obj.timestamps[:])
                        OK(f'BehavioralEvents[\'{key}\'] : {len(ts)} timestamps')
                        found_any = True
                except Exception:
                    pass

                if not found_any:
                    WRN('Aucun signal de lick trouvé dans behavior.')

            except Exception as e:
                ERR(f'Erreur accès behavior : {e}')

    except FileNotFoundError:
        ERR(f'FICHIER INTROUVABLE : {path}')
    except Exception as e:
        ERR(f'ERREUR LECTURE : {e}')

print(f'\n{BLD}{"="*72}{RST}')
print(f'{BLD}Vérification terminée — {len(SESSION_PATHS)} session(s).{RST}')




  data/brut/AO027_20181101_134512.nwb

[1] Aires disponibles & nombre de neurones
    Unités totales : 78
    Colonne aire   : Target_area
    2 aire(s) :
        mPFC                                 52 neurones
      ★ wS1                                  26 neurones
    (★ = probable wS1 / SSp-bfd)

[2] Structure des stimuli
    Trials totaux : 402

    Whisker stim  : 4 amplitudes → [1. 2. 3. 4.]
    Trials whisker : 220
      amp 1 :    56 trials
      amp 2 :    61 trials
      amp 3 :    57 trials
      amp 4 :    46 trials
    Catch         : 182 trials  (colonne 'no_stim')

    Toutes les colonnes trials : ['start_time', 'stop_time', 'trial_type', 'whisker_stim', 'whisker_stim_amplitude', 'whisker_stim_time', 'whisker_stim_duration', 'no_stim', 'no_stim_time', 'reward_available', 'response_window_start_time', 'response_window_stop_time', 'perf', 'lick_time', 'jaw_dlc_licks', 'lick_flag']

[3] Lick times disponibles
    PiezoLickSignal (continu) : 5385010 pts @ ~1000 Hz
    Beh

In [30]:
# ── Filtre : sessions avec wS1/SSp-bfd ≥ 10 neurones + résumé stimuli ──────
# Critères :
#   ✓ wS1 présent (Target_area == 'wS1'  OU  ccf_parent_acronym == 'SSp-bfd'
#                  OU aire in {'BC', 'S1BF'})
#   ✓ ≥ 10 neurones dans cette aire
# Affiche aussi : type de stim (amp1-4 vs unique) + lick dispo

import numpy as np
from pynwb import NWBHDF5IO
from collections import Counter

BLD = '\033[1m'; GRN = '\033[92m'; YLW = '\033[93m'; RED = '\033[91m'; RST = '\033[0m'
WSS = {'SSp-bfd', 'wS1', 'BC', 'S1BF'}          # synonymes wS1
MIN_NEURONS = 10

valid, invalid = [], []

for path in SESSION_PATHS:
    name = path.split('/')[-1]
    row  = {'path': path, 'name': name,
            'ws1_area': None, 'n_ws1': 0,
            'stim_type': '?', 'amps': [],
            'lick': '?', 'error': None}
    try:
        with NWBHDF5IO(path, 'r') as io:
            nwb = io.read()

            # ── Aires ────────────────────────────────────────────────────
            units_df = nwb.units.to_dataframe()
            for col in ('Target_area', 'ccf_parent_acronym', 'ccf_acronym', 'brain_area', 'location'):
                if col in units_df.columns:
                    area_col = col; break
            else:
                area_col = None

            if area_col:
                raw   = units_df[area_col].values
                areas = np.array([a.decode() if isinstance(a, (bytes, bytearray)) else str(a)
                                   for a in raw])
                counts = Counter(areas)
                # cherche la meilleure aire wS1
                for candidate in ('wS1', 'SSp-bfd', 'BC', 'S1BF'):
                    if candidate in counts:
                        row['ws1_area'] = candidate
                        row['n_ws1']    = counts[candidate]
                        break
                # fallback : toute aire contenant 'SSp'
                if row['ws1_area'] is None:
                    for aire, n in counts.items():
                        if 'SSp' in aire:
                            row['ws1_area'] = aire
                            row['n_ws1']    = n
                            break

            # ── Stimuli ──────────────────────────────────────────────────
            tr = nwb.trials.to_dataframe()
            if 'whisker_stim_amplitude' in tr.columns:
                vals = tr['whisker_stim_amplitude'].dropna()
                amps = np.sort(vals[vals > 0].unique())
                row['amps']      = list(amps.astype(int))
                row['stim_type'] = f'{len(amps)} amp(s)' if len(amps) > 1 else 'unique'
            elif 'whisker_stim' in tr.columns:
                row['stim_type'] = 'unique (binaire)'
                row['amps']      = [1]
            elif 'whisker_stim_time' in tr.columns:
                row['stim_type'] = 'unique (time only)'
                row['amps']      = [1]
            else:
                row['stim_type'] = 'INCONNU'

            # ── Lick ─────────────────────────────────────────────────────
            lick_src = []
            try:
                beh = nwb.processing['behavior']
                try:
                    beh['BehavioralTimeSeries']['PiezoLickSignal']
                    lick_src.append('Piezo')
                except Exception:
                    pass
                try:
                    be = beh['BehavioralEvents']
                    for k in be.time_series:
                        lick_src.append(k)
                except Exception:
                    pass
            except Exception:
                pass
            row['lick'] = ', '.join(lick_src) if lick_src else 'AUCUN'

    except FileNotFoundError:
        row['error'] = 'FICHIER INTROUVABLE'
    except Exception as e:
        row['error'] = str(e)[:60]

    if row['error']:
        invalid.append(row)
    elif row['n_ws1'] >= MIN_NEURONS:
        valid.append(row)
    else:
        invalid.append(row)

# ── Affichage ────────────────────────────────────────────────────────────────
print(f'\n{BLD}{"="*72}{RST}')
print(f'{BLD}  ✓ Sessions VALIDES  (wS1 ≥ {MIN_NEURONS} neurones) — {len(valid)}/{len(SESSION_PATHS)}{RST}')
print(f'{BLD}{"="*72}{RST}')

col_w = [38, 12, 18, 10]
header = f"  {'Session':<{col_w[0]}} {'n wS1':>{col_w[1]}} {'Stim':^{col_w[2]}} {'Lick':<{col_w[3]}}"
print(f'{BLD}{header}{RST}')
print('  ' + '-'*(sum(col_w)+3))

for r in valid:
    aire_str  = f"{r['ws1_area']} ({r['n_ws1']})"
    stim_str  = r['stim_type']
    if r['amps'] and len(r['amps']) > 1:
        stim_str += f"  [{','.join(str(a) for a in r['amps'])}]"
    lick_ok   = 'AUCUN' not in r['lick']
    lick_str  = (GRN if lick_ok else RED) + r['lick'] + RST
    print(f"  {GRN}{r['name']:<{col_w[0]}}{RST} {aire_str:>{col_w[1]}}  {stim_str:<{col_w[2]}} {lick_str}")

if not valid:
    print(f'  {YLW}(aucune session ne passe le filtre){RST}')

print(f'\n{BLD}{"="*72}{RST}')
print(f'{BLD}  ✗ Sessions INVALIDES / ignorées — {len(invalid)}{RST}')
print(f'{BLD}{"="*72}{RST}')

for r in invalid:
    if r['error']:
        print(f"  {RED}✗ {r['name']:<40} {r['error']}{RST}")
    else:
        ws1_info = f"{r['ws1_area']} ({r['n_ws1']})" if r['ws1_area'] else 'pas de wS1'
        print(f"  {YLW}✗ {r['name']:<40} {ws1_info:<20} stim={r['stim_type']}{RST}")

# ── Chemins copiables pour results.ipynb ─────────────────────────────────────
if valid:
    print(f'\n{BLD}── SESSION_PATHS à copier dans results.ipynb ──{RST}')
    print('SESSION_PATHS = [')
    for r in valid:
        print(f"    '{r['path']}',")
    print(']')


  ✓ Sessions VALIDES  (wS1 ≥ 10 neurones) — 18/25
  Session                                       n wS1        Stim        Lick      
  ---------------------------------------------------------------------------------
  AO027_20181101_134512.nwb                  wS1 (26)  4 amp(s)  [1,2,3,4] Piezo, EngagedTrials, ReactionTimes, ResponseType, StimFlags, TrialOnsets, VideoOnsets, correct_rejection_trial, false_alarm_trial, jaw_dlc_licks, whisker_hit_trial, whisker_miss_trial
  AO028_20181102_110339.nwb                  wS1 (30)  4 amp(s)  [1,2,3,4] Piezo, EngagedTrials, ReactionTimes, ResponseType, StimFlags, TrialOnsets, VideoOnsets, correct_rejection_trial, false_alarm_trial, jaw_dlc_licks, whisker_hit_trial, whisker_miss_trial
  AO028_20181103_83222.nwb                   wS1 (28)  4 amp(s)  [1,2,3,4] Piezo, EngagedTrials, ReactionTimes, ResponseType, StimFlags, TrialOnsets, VideoOnsets, correct_rejection_trial, false_alarm_trial, jaw_dlc_licks, whisker_hit_trial, whisker_miss_trial
 

In [31]:
# ── Tri : valid → data/valid/   |   non valide → data/non_valid/ ──────────
import shutil, os

VALID_DIR   = 'data/valid'
INVALID_DIR = 'data/non_valid'
os.makedirs(VALID_DIR,   exist_ok=True)
os.makedirs(INVALID_DIR, exist_ok=True)

BLD = '\033[1m'; GRN = '\033[92m'; YLW = '\033[93m'; RED = '\033[91m'; RST = '\033[0m'

def move_file(src, dest_dir, label_color):
    name = os.path.basename(src)
    dst  = os.path.join(dest_dir, name)
    if not os.path.exists(src):
        print(f'  {RED}✗ introuvable  {name}{RST}')
        return 'error'
    if os.path.exists(dst):
        print(f'  {YLW}⊘ déjà présent {name}  →  {dest_dir}/{RST}')
        return 'skip'
    try:
        shutil.move(src, dst)
        print(f'  {label_color}✓ déplacé      {name}  →  {dest_dir}/{RST}')
        return 'ok'
    except Exception as e:
        print(f'  {RED}✗ erreur       {name}  →  {e}{RST}')
        return 'error'

print(f'{BLD}── Sessions VALIDES → {VALID_DIR}/ ──────────────────────────{RST}')
v_ok = v_skip = v_err = 0
for r in valid:
    res = move_file(r['path'], VALID_DIR, GRN)
    if res == 'ok':   v_ok   += 1
    elif res == 'skip': v_skip += 1
    else:             v_err  += 1

print(f'\n{BLD}── Sessions INVALIDES → {INVALID_DIR}/ ───────────────────────{RST}')
i_ok = i_skip = i_err = 0
for r in invalid:
    if r['error'] and 'INTROUVABLE' in str(r['error']):
        continue   # déjà absent, on saute
    res = move_file(r['path'], INVALID_DIR, YLW)
    if res == 'ok':   i_ok   += 1
    elif res == 'skip': i_skip += 1
    else:             i_err  += 1

print(f'\n{BLD}─── Résumé ───────────────────────────────────────────────{RST}')
print(f'  → {VALID_DIR}/   : {GRN}{v_ok} déplacé(s){RST}  {YLW}{v_skip} déjà présent(s){RST}  {RED}{v_err} erreur(s){RST}')
print(f'  → {INVALID_DIR}/ : {YLW}{i_ok} déplacé(s){RST}  {YLW}{i_skip} déjà présent(s){RST}  {RED}{i_err} erreur(s){RST}')

# ── SESSION_PATHS prêt pour results.ipynb ─────────────────────────────────
all_valid_files = sorted([
    os.path.join(VALID_DIR, f) for f in os.listdir(VALID_DIR) if f.endswith('.nwb')
])
if all_valid_files:
    print(f'\n{BLD}SESSION_PATHS pour results.ipynb :{RST}')
    print('SESSION_PATHS = [')
    for p in all_valid_files:
        print(f"    '{p}',")
    print(']')

── Sessions VALIDES → data/valid/ ──────────────────────────
  ✓ déplacé      AO027_20181101_134512.nwb  →  data/valid/
  ✓ déplacé      AO028_20181102_110339.nwb  →  data/valid/
  ✓ déplacé      AO028_20181103_83222.nwb  →  data/valid/
  ✓ déplacé      AO036_20190507_183647.nwb  →  data/valid/
  ✓ déplacé      AO039_20190627_151857.nwb  →  data/valid/
  ✓ déplacé      AO040_20190709_102521.nwb  →  data/valid/
  ✓ déplacé      AO049_20191009_160435.nwb  →  data/valid/
  ✓ déplacé      AO050_20191007_195438.nwb  →  data/valid/
  ✓ déplacé      AO051_20191004_182709.nwb  →  data/valid/
  ✓ déplacé      AO065_20201023_100155.nwb  →  data/valid/
  ✓ déplacé      AO066_20201023_180037.nwb  →  data/valid/
  ✓ déplacé      AO067_20201022_190740.nwb  →  data/valid/
  ✓ déplacé      AO074_20201222_164350.nwb  →  data/valid/
  ✓ déplacé      AO075_20201223_134131.nwb  →  data/valid/
  ✓ déplacé      AO076_20201222_212759.nwb  →  data/valid/
  ✓ déplacé      AO077_20201223_91923.nwb  →  data/vali